# Quantum-Chemical Geometries, Electronic Descriptors, and Li+ Binding Energies for Salt Anions and Solvent Molecules Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.8pha-319z/fair2.json`

The dataset describes computational chemistry results for salt anions and solvent molecules, including optimized geometries, electronic descriptors, and Li+ binding energies.

> **Important note**: In Croissant datasets, entities such as record sets and fields should be referenced by their `@id`. This notebook is written to discover those IDs dynamically from the metadata and then use them consistently.

In [ ]:
# Ensure the required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the dataset metadata from the Croissant URL and inspect the high-level dataset description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.8pha-319z/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display high-level metadata safely without treating metadata as a dictionary-like iterable
metadata_json = metadata.to_json()
print("Dataset name:", metadata_json.get("name"))
print("Description:", metadata_json.get("description"))
print("Version:", metadata_json.get("version"))
print("License:", metadata_json.get("license"))
print("Citation:", metadata_json.get("citeAs"))

## 2. Data Overview
In this section, we inspect the Croissant metadata to discover available record sets and their fields.

Because all dataset entities should be referenced by `@id`, we first extract the discovered record set IDs and field IDs dynamically from the metadata.

In [ ]:
metadata_json = metadata.to_json()
record_sets_meta = metadata_json.get("recordSet", [])

print(f"Number of discovered record sets: {len(record_sets_meta)}")

record_set_overview = []
for rs in record_sets_meta:
    rs_id = rs.get("@id")
    rs_name = rs.get("name")
    rs_description = rs.get("description")
    rs_fields = rs.get("field", [])
    record_set_overview.append({
        "record_set_id": rs_id,
        "record_set_name": rs_name,
        "field_count": len(rs_fields),
        "description": rs_description,
    })

record_set_overview_df = pd.DataFrame(record_set_overview)
record_set_overview_df

### Record set and field inspection
The next cell prints each record set `@id` and the associated field `@id`s. These are the IDs you should use in downstream loading and analysis.

In [ ]:
for rs in record_sets_meta:
    rs_id = rs.get("@id")
    rs_name = rs.get("name")
    print("\n" + "=" * 100)
    print(f"Record set name: {rs_name}")
    print(f"Record set @id: {rs_id}")
    print("Fields:")
    for field in rs.get("field", []):
        print(f"  - field @id: {field.get('@id')} | name: {field.get('name')} | dataType: {field.get('dataType')}")

### Preview records from a discovered record set
This cell uses a variable to select a record set by `@id` and prints a few records. If the dataset contains multiple record sets, we default to the first one for preview.

In [ ]:
record_set_ids = [rs.get("@id") for rs in record_sets_meta if rs.get("@id")]
preview_record_set_id = record_set_ids[0] if record_set_ids else None

print("Preview record set @id:", preview_record_set_id)

if preview_record_set_id is not None:
    for i, record in enumerate(dataset.records(record_set=preview_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets were declared in the Croissant metadata.")

## 3. Data Extraction
Now we load each discovered record set into a pandas DataFrame.

This section is written defensively so it still runs even if the dataset exposes no record sets directly in the metadata.

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

print("\nLoaded record set IDs:")
print(list(dataframes.keys()))

In [ ]:
# Show columns and a preview for the first successfully loaded DataFrame
loaded_record_set_ids = list(dataframes.keys())
analysis_record_set_id = loaded_record_set_ids[0] if loaded_record_set_ids else None

print("Analysis record set @id:", analysis_record_set_id)

if analysis_record_set_id is not None:
    print("Columns:")
    print(dataframes[analysis_record_set_id].columns.tolist())
    display(dataframes[analysis_record_set_id].head())
else:
    print("No record sets were loaded into DataFrames.")

### Helper functions for field discovery
To make the notebook reusable, we define helper functions that identify likely numeric and grouping fields based on loaded DataFrame content and metadata.

In [ ]:
def get_record_set_metadata(record_sets, record_set_id):
    for rs in record_sets:
        if rs.get("@id") == record_set_id:
            return rs
    return None

def get_field_ids(record_set_meta):
    if not record_set_meta:
        return []
    return [field.get("@id") for field in record_set_meta.get("field", []) if field.get("@id")]

def find_numeric_columns(df):
    if df is None or df.empty:
        return []
    numeric_cols = []
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().sum() > 0:
            numeric_cols.append(col)
    return numeric_cols

def find_categorical_columns(df, max_unique=25):
    if df is None or df.empty:
        return []
    categorical_cols = []
    for col in df.columns:
        if df[col].dtype == "object":
            nunique = df[col].nunique(dropna=True)
            if 1 < nunique <= max_unique:
                categorical_cols.append(col)
    return categorical_cols

## 4. Exploratory Data Analysis (EDA)
We now perform common data processing steps:

- identify a numeric field
- filter records using a dynamic threshold
- normalize the numeric field
- optionally group by a categorical field
- summarize the processed data

All field references below use DataFrame column names that originate from dataset field `@id`s.

In [ ]:
if analysis_record_set_id is not None:
    analysis_df = dataframes[analysis_record_set_id].copy()
    analysis_record_set_meta = get_record_set_metadata(record_sets_meta, analysis_record_set_id)
    available_field_ids = get_field_ids(analysis_record_set_meta)
    
    print("Available field @ids for analysis record set:")
    print(available_field_ids)
    
    numeric_fields = find_numeric_columns(analysis_df)
    categorical_fields = find_categorical_columns(analysis_df)
    
    print("\nCandidate numeric field @ids / columns:")
    print(numeric_fields)
    
    print("\nCandidate grouping field @ids / columns:")
    print(categorical_fields)
else:
    print("No analysis DataFrame is available.")

In [ ]:
if analysis_record_set_id is not None and not analysis_df.empty:
    numeric_field_id = numeric_fields[0] if numeric_fields else None
    group_field_id = categorical_fields[0] if categorical_fields else None
    
    print("Selected numeric field @id:", numeric_field_id)
    print("Selected grouping field @id:", group_field_id)
    
    if numeric_field_id is not None:
        analysis_df[numeric_field_id] = pd.to_numeric(analysis_df[numeric_field_id], errors="coerce")
        numeric_series = analysis_df[numeric_field_id].dropna()
        
        if len(numeric_series) > 0:
            threshold = numeric_series.median()
            filtered_df = analysis_df[analysis_df[numeric_field_id] > threshold].copy()
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())
            
            std_value = filtered_df[numeric_field_id].std()
            if pd.notna(std_value) and std_value != 0:
                normalized_field_id = f"{numeric_field_id}_normalized"
                filtered_df[normalized_field_id] = (
                    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
                ) / std_value
                print(f"Normalized field created: {normalized_field_id}")
                display(filtered_df[[numeric_field_id, normalized_field_id]].head())
            else:
                normalized_field_id = None
                print("Standard deviation is zero or undefined; normalization skipped.")
            
            if group_field_id is not None and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(["count", "mean", "median", "min", "max"]).sort_values("mean", ascending=False)
                print(f"Grouped summary by {group_field_id}:")
                display(grouped_df.head(10))
            else:
                grouped_df = None
                print("No suitable grouping field was found.")
        else:
            filtered_df = pd.DataFrame()
            normalized_field_id = None
            grouped_df = None
            print("No non-null numeric values available for filtering.")
    else:
        filtered_df = pd.DataFrame()
        normalized_field_id = None
        grouped_df = None
        print("No numeric field was identified.")
else:
    filtered_df = pd.DataFrame()
    normalized_field_id = None
    grouped_df = None
    print("Analysis DataFrame is not available or is empty.")

### Optional outlier trimming
A common preprocessing step is to remove extreme values. The following cell trims the selected numeric field using the interquartile range (IQR) rule.

In [ ]:
if analysis_record_set_id is not None and 'numeric_field_id' in globals() and numeric_field_id is not None:
    clean_df = analysis_df.copy()
    x = pd.to_numeric(clean_df[numeric_field_id], errors="coerce")
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    trimmed_df = clean_df[(x >= lower_bound) & (x <= upper_bound)].copy()
    
    print("Numeric field @id:", numeric_field_id)
    print(f"IQR lower bound: {lower_bound}")
    print(f"IQR upper bound: {upper_bound}")
    print(f"Original rows: {len(clean_df)}")
    print(f"Trimmed rows: {len(trimmed_df)}")
    display(trimmed_df.head())
else:
    trimmed_df = pd.DataFrame()
    print("Outlier trimming skipped because no numeric field is available.")

## 5. Visualization
We create a few simple visualizations for the selected numeric field and, when possible, compare it across a grouping field.

In [ ]:
if analysis_record_set_id is not None and 'numeric_field_id' in globals() and numeric_field_id is not None:
    plot_df = analysis_df.copy()
    plot_df[numeric_field_id] = pd.to_numeric(plot_df[numeric_field_id], errors="coerce")
    plot_df = plot_df.dropna(subset=[numeric_field_id])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.histplot(plot_df[numeric_field_id], kde=True, ax=axes[0], color="steelblue")
    axes[0].set_title(f"Distribution of {numeric_field_id}")
    axes[0].set_xlabel(numeric_field_id)
    
    sns.boxplot(x=plot_df[numeric_field_id], ax=axes[1], color="orange")
    axes[1].set_title(f"Boxplot of {numeric_field_id}")
    axes[1].set_xlabel(numeric_field_id)
    
    plt.tight_layout()
    plt.show()
else:
    print("Numeric visualization skipped because no numeric field is available.")

In [ ]:
if (
    analysis_record_set_id is not None
    and 'numeric_field_id' in globals() and numeric_field_id is not None
    and 'group_field_id' in globals() and group_field_id is not None
    and group_field_id in analysis_df.columns
):
    comparison_df = analysis_df.copy()
    comparison_df[numeric_field_id] = pd.to_numeric(comparison_df[numeric_field_id], errors="coerce")
    comparison_df = comparison_df.dropna(subset=[numeric_field_id, group_field_id])
    
    top_groups = comparison_df[group_field_id].value_counts().head(10).index
    comparison_df = comparison_df[comparison_df[group_field_id].isin(top_groups)]
    
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=comparison_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Grouped visualization skipped because a suitable grouping field is not available.")

### Correlation analysis for numeric fields
If multiple numeric fields are available in the selected record set, a correlation heatmap can help reveal relationships among descriptors.

In [ ]:
if analysis_record_set_id is not None:
    numeric_df = analysis_df.copy()
    numeric_candidates = find_numeric_columns(numeric_df)
    
    for col in numeric_candidates:
        numeric_df[col] = pd.to_numeric(numeric_df[col], errors="coerce")
    
    numeric_df = numeric_df[numeric_candidates]
    numeric_df = numeric_df.dropna(axis=1, how="all")
    
    if numeric_df.shape[1] >= 2:
        corr = numeric_df.corr(numeric_only=True)
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr, cmap="coolwarm", center=0)
        plt.title("Correlation heatmap of numeric field @ids")
        plt.tight_layout()
        plt.show()
    else:
        print("Correlation heatmap skipped because fewer than two numeric fields are available.")
else:
    print("Correlation analysis skipped because no analysis DataFrame is available.")

## 6. Conclusion
In this notebook, we:

- loaded a Croissant dataset using `mlcroissant`
- inspected metadata and discovered record set and field `@id`s
- loaded available record sets into pandas DataFrames
- selected numeric and grouping fields dynamically
- filtered, normalized, and optionally grouped the data
- visualized distributions and relationships among fields

This workflow is especially useful when working with Croissant datasets because the schema can be explored programmatically and reused in reproducible downstream analysis.

If you want to extend this notebook, useful next steps include:

- selecting a domain-specific field `@id` such as a binding energy or descriptor column
- merging multiple record sets when the metadata defines linked entities
- building predictive models from the discovered numeric descriptors
- tracing any selected feature back to its exact Croissant field `@id` for reproducibility